In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
import pandas as pd 


In [3]:
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')

In [4]:
train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [5]:
train.describe()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,54.136706,0.714735,3.312752,130.497433,245.011814,0.079987,0.981660,152.816763,0.273725,0.716028,1.455871,0.451040,4.618873
std,181865.479132,8.256301,0.451541,0.851615,14.975802,33.681581,0.271274,0.998783,19.112927,0.445870,0.948472,0.545192,0.798549,1.950007
min,0.000000,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,157499.750000,48.000000,0.000000,3.000000,120.000000,223.000000,0.000000,0.000000,142.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,314999.500000,54.000000,1.000000,4.000000,130.000000,243.000000,0.000000,0.000000,157.000000,0.000000,0.100000,1.000000,0.000000,3.000000
75%,472499.250000,60.000000,1.000000,4.000000,140.000000,269.000000,0.000000,2.000000,166.000000,1.000000,1.400000,2.000000,1.000000,7.000000
max,629999.000000,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000


In [6]:
train.isnull().sum()

id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = train.select_dtypes(include='object').columns

preprocess = ColumnTransformer([
    ('one_hot', OneHotEncoder(), cat_cols)
], remainder='passthrough')


In [8]:
train_clean = train.dropna(subset=["Heart Disease"])

In [9]:
X = train_clean.drop(["id", "Heart Disease"], axis=1)
y = train_clean["Heart Disease"]

In [10]:
print("NaNs in X:", X.isna().sum().sum())


NaNs in X: 0


In [11]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)


Train shape: (504000, 13)
Validation shape: (126000, 13)


In [12]:
print(train.columns)

Index(['id', 'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
       'Heart Disease'],
      dtype='object')


In [13]:
print("NaNs in X:", X.isna().sum().sum())
print("NaNs in y:", y.isna().sum())


NaNs in X: 0
NaNs in y: 0


In [14]:
print(y.value_counts())

Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64


In [15]:
y = train_clean["Heart Disease"].map({
    "Absence": 0,
    "Presence": 1
})

x=train_clean.drop(["id", "Heart Disease"], axis=1)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [17]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])


log_model.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index([], dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

In [18]:
from sklearn.metrics import roc_auc_score

preds = log_model.predict_proba(x_val)[:,1]
print("Logistic AUC:", roc_auc_score(y_val, preds))

Logistic AUC: 0.951546958953206


In [19]:
import pandas as pd

temp = X.copy()
temp["target"] = y

corr = temp.corr(numeric_only=True)["target"].sort_values(ascending=False)
print(corr)


target                     1.000000
Thallium                   0.605776
Chest pain type            0.460684
Exercise angina            0.441864
Number of vessels fluro    0.438604
ST depression              0.430641
Slope of ST                0.415050
Sex                        0.342446
EKG results                0.218961
Age                        0.212091
Cholesterol                0.082753
FBS over 120               0.033570
BP                        -0.005181
Max HR                    -0.440985
Name: target, dtype: float64


In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    log_model,
    X,
    y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("Fold AUCs:", scores)
print("Mean CV AUC:", scores.mean())


Fold AUCs: [0.95075056 0.94996061 0.95078322 0.95021808 0.95073789]
Mean CV AUC: 0.9504900722631842


In [21]:
from lightgbm import LGBMClassifier

lgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    )
])

lgb_model.fit(X, y)

scores = cross_val_score(lgb_model, x, y, cv=cv, scoring="roc_auc", n_jobs=-1)

[LightGBM] [Info] Number of positive: 282454, number of negative: 347546
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 423
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448340 -> initscore=-0.207381
[LightGBM] [Info] Start training from score -0.207381


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
cat_features = X.select_dtypes(include=['object']).columns.tolist()
print("Categorical features:", cat_features)


Categorical features: []


In [23]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_state=42,
    verbose=100,
    early_stopping_rounds=100
)

cat_model.fit(x_train, y_train, eval_set=(x_val, y_val))


0:	test: 0.9398405	best: 0.9398405 (0)	total: 133ms	remaining: 1m 46s
100:	test: 0.9547495	best: 0.9547495 (100)	total: 6.92s	remaining: 47.9s
200:	test: 0.9552510	best: 0.9552510 (200)	total: 13.6s	remaining: 40.6s
300:	test: 0.9556032	best: 0.9556034 (299)	total: 20.1s	remaining: 33.4s
400:	test: 0.9558815	best: 0.9558815 (400)	total: 26.7s	remaining: 26.6s
500:	test: 0.9560498	best: 0.9560502 (499)	total: 33.3s	remaining: 19.9s
600:	test: 0.9561291	best: 0.9561291 (600)	total: 39.8s	remaining: 13.2s
700:	test: 0.9561783	best: 0.9561783 (700)	total: 46.4s	remaining: 6.56s
799:	test: 0.9562225	best: 0.9562227 (798)	total: 53s	remaining: 0us

bestTest = 0.956222737
bestIteration = 798

Shrink model to first 799 iterations.


In [24]:
from sklearn.metrics import roc_auc_score

cat_preds = cat_model.predict_proba(x_val)[:,1]
cat_auc = roc_auc_score(y_val, cat_preds)
print("CatBoost Validation AUC:", cat_auc)


CatBoost Validation AUC: 0.9562227369998779


In [25]:
x_full = train.drop(["id", "Heart Disease"], axis=1)
y_full = train["Heart Disease"]


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import pandas as pd

train = train.dropna(subset=["Heart Disease"])

x = train.drop(["id", "Heart Disease"], axis=1)
y = train["Heart Disease"]
test_x = test.drop("id", axis=1)

num_imputer = SimpleImputer(strategy="median")
x_full = pd.DataFrame(num_imputer.fit_transform(x), columns=x.columns)
test_full = pd.DataFrame(num_imputer.transform(test_x), columns=test_x.columns)

x_train, x_val, y_train, y_val = train_test_split(x_full, y, test_size=0.2, stratify=y, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(x_train, y_train)
log_val_preds = log_model.predict_proba(x_val)[:, 1]
print("Logistic AUC:", roc_auc_score(y_val, log_val_preds))

lgb_model = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8)
lgb_model.fit(x_train, y_train)
lgb_val_preds = lgb_model.predict_proba(x_val)[:, 1]
print("LightGBM AUC:", roc_auc_score(y_val, lgb_val_preds))

cat_model = CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6, eval_metric='AUC', random_state=42, verbose=0, early_stopping_rounds=100)
cat_model.fit(x_train, y_train, eval_set=(x_val, y_val))
cat_val_preds = cat_model.predict_proba(x_val)[:, 1]
print("CatBoost AUC:", roc_auc_score(y_val, cat_val_preds))

log_test_preds = log_model.predict_proba(test_full)[:, 1]
lgb_test_preds = lgb_model.predict_proba(test_full)[:, 1]
cat_test_preds = cat_model.predict_proba(test_full)[:, 1]

final_preds = (log_test_preds + lgb_test_preds + cat_test_preds) / 3
submission = pd.DataFrame({"id": test["id"], "Heart Disease": final_preds})
submission.to_csv("submission.csv", index=False)

Logistic AUC: 0.9515479554812951
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.075191 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 416
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warn